In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# Load data
df = pd.read_csv("/signals.csv")

# Convert date and sort chronologically
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Keep first 49 SMA crossover values as NaN
df.loc[:48, "sma_crossover"] = np.nan

valid_sma = df["10ma"].notna() & df["50ma"].notna()

df.loc[valid_sma, "sma_crossover"] = (
    df.loc[valid_sma, "10ma"] > df.loc[valid_sma, "50ma"]
).astype(int)

# Keep last 5 target values as NaN
df.loc[df["future_5d_return"].isna(), "target"] = np.nan

# Define predictors and target
signal_cols = [
    "1d_return",
    "5d_return",
    "20d_return",
    "percent_ma_gap",
    "sma_crossover",
    "20d_drawdown",
    "10d_volatility",
    "volatility_ratio",
    "volume_zscore",
    "rsi_14"
]

target = "target"

# Keep full corrected dataset for predictions
predictions_base = df.copy()

# Keep only relevant columns for modeling
model_df = df[["date"] + signal_cols + [target]].copy()

# Remove missing values for model training
model_df = model_df.dropna(
    subset=signal_cols + [target]
).reset_index(drop=True)

print(model_df.shape)
print(model_df.columns)

(1201, 12)
Index(['date', '1d_return', '5d_return', '20d_return', 'percent_ma_gap',
       'sma_crossover', '20d_drawdown', '10d_volatility', 'volatility_ratio',
       'volume_zscore', 'rsi_14', 'target'],
      dtype='object')


In [ ]:
import os
import pandas as pd
import statsmodels.api as sm

# 1. Create output folders
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

# 2. Chronological Split (80% Train/Dev, 20% Holdout Test)
split_idx = int(len(model_df) * 0.80)

train_df = model_df.iloc[:split_idx].copy()
test_df = model_df.iloc[split_idx:].copy()

# 3. Fit Statsmodels Logistic Regression on Training Data ONLY
X_train = train_df[signal_cols]
X_train_const = sm.add_constant(X_train)  # Includes intercept
y_train = train_df[target]

logit_model = sm.Logit(y_train, X_train_const).fit()

# Print summary to verify model fit
print(logit_model.summary())

# 4. Save results/regression_coefficients.csv
coeff_df = pd.DataFrame({
    "feature": logit_model.params.index,
    "coefficient": logit_model.params.values,
    "std_err": logit_model.bse.values,
    "z_stat": logit_model.tvalues.values,
    "p_value": logit_model.pvalues.values,
    "conf_lower": logit_model.conf_int()[0].values,
    "conf_upper": logit_model.conf_int()[1].values,
})
coeff_df.to_csv("results/regression_coefficients.csv", index=False)

# 5. Save results/model_predictions.csv
X_full = model_df[signal_cols]
X_full_const = sm.add_constant(X_full)

predictions_df = model_df.copy()
predictions_df["predicted_prob"] = logit_model.predict(X_full_const)
predictions_df["sample_split"] = "train_dev"
predictions_df.loc[split_idx:, "sample_split"] = "test_holdout"

predictions_df.to_csv("results/model_predictions.csv", index=False)

# 6. Generate statistics values (standard errors, p-values, etc.)
doc_text = f"""# Baseline Logistic Regression Model Notes

## Model Summary
{logit_model.summary().as_text()}
"""


Optimization terminated successfully.
         Current function value: 0.671632
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:                  960
Model:                          Logit   Df Residuals:                      949
Method:                           MLE   Df Model:                           10
Date:                Sun, 16 Aug 2026   Pseudo R-squ.:                0.008492
Time:                        02:45:45   Log-Likelihood:                -644.77
converged:                       True   LL-Null:                       -650.29
Covariance Type:            nonrobust   LLR p-value:                    0.3540
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const                1.4950      0.678      2.206      0.027       0.167       2.823
1d_return  